# Multiagent Pipeline

In [491]:
# Facoltativo: installa dipendenze se ti servono in un ambiente pulito
# %pip install -r requirements.txt

In [492]:
import pandas as pd
df = pd.read_csv('data/raw/ALLARMI.csv')
df

,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,...,ZONA,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli
0,Voli con Allarmi,FCO,IST,2024,01,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,...,5,1,Manuale,NaN,NaN,Turchia,ITA,5,Italia,1
1,Viaggiatori con Allarmi,CIA,STN,2024,02,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,...,5,5,Manuale,NaN,NaN,Regno Unito,ITA,5,Italia,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,01,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,...,5,110,TSC,NaN,NaN,Regno Unito,ITA,5,Italia,110
3,Voli con Allarmi,MXP,LHR,2024,02,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,...,2,1,SDI,NaN,NaN,Regno Unito,ITA,2,Italia,1
4,Viaggiatori con Allarmi,PSA,BRS,2024,02,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,...,8,2,INTERPOL,NaN,NaN,Regno Unito,ITA,8,Italia,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5075,Nulla a procedere SDI,MXP,LTN,2023,01,2024-01-19 17:30:00,Malpensa,London Luton,Milano,Londra,...,2,5,TSC,NaN,NaN,Regno Unito,ITA,2,Italia,5
5076,Voli disponibili in ingresso al Sistema,BLQ,RAK,2024,02,FEB 06 2024,Guglielmo Marconi,Menara,Bologna,Marrakech,...,8,1,TSC,NaN,NaN,Marocco,ITA,8,Italia,1
5077,Voli con Allarmi,FCO,DOH,2024,01,2024-01-12 15:35:00,Fiumicino,Hamad International,Roma,Doha,...,5,1,NSIS,NaN,NaN,Qatar,ITA,5,Italia,1
5078,"Voli solo visualizzati, ma NON investigati",LIN,LHR,2024,01,2024-01-13 11:50:00,Linate,London Heathrow,Milano,Londra,...,2,1,TSC,NaN,NaN,Regno Unito,ITA,2,Italia,1


# Configuration

In [493]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import sys
import io
import traceback

load_dotenv()

MISTRAL_BASE_URL = "https://api.mistral.ai/v1"
MISTRAL_API_KEY  = os.getenv("MISTRAL_API_KEY", "")
MISTRAL_MODEL    = "mistral-small-latest"

_client = OpenAI(base_url=MISTRAL_BASE_URL, api_key=MISTRAL_API_KEY)


# Paths 
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
ALLARMI_CSV = os.path.join(RAW_DIR, "ALLARMI.csv")
TIPOLOGIA_CSV = os.path.join(RAW_DIR, "TIPOLOGIA_VIAGGIATORE.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
FINDINGS_JSON = os.path.join(OUTPUT_DIR, "findings.json")
COLUMN_PROFILES_JSON = os.path.join(OUTPUT_DIR, "column_profiles.json")
CLEANING_PLAN_JSON   = os.path.join(OUTPUT_DIR, "cleaning_plan.json")   # NEW: slim JSON for cleaning agent

# Profiling 
DOMINANT_FORMAT_SAMPLE_SIZE = 200   # NEW: how many rows to inspect when detecting the dominant format

In [494]:
# CLEANING HELPERS — generic, hard-coded-free
# Reusable building blocks for the cleaning agent. They are completely
# generic: no column-specific knowledge, no domain dictionaries, no
# regex tied to a specific data source.

DEFAULT_CLEANING_MISSING_TOKENS = {
    "", " ", "na", "n/a", "null", "none", "nan", "unknown", "undefined",
    "-", "--", "not available", "missing", "n.d.", "nd"
}

import re
import json
import unicodedata
from typing import Optional, Iterable
import pandas as pd



def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def _collapse_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


# ── Column name normalization ─────────────────────────────────────────

def to_snake_case(name: str) -> str:
    """
    Convert a column name to lowercase snake_case in a stable way.
    Handles spaces, punctuation, camelCase, PascalCase and noisy symbols.
    """
    name = "" if name is None else str(name)
    name = _strip_accents(name).strip()

    # Separate camelCase / PascalCase boundaries
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", name)

    # Replace non-alphanumeric runs with underscore
    name = re.sub(r"[^0-9A-Za-z]+", "_", name)

    # Collapse underscores and trim
    name = re.sub(r"_+", "_", name).strip("_").lower()

    if not name:
        name = "unnamed_column"

    # If the name starts with a digit, prefix it to keep it identifier-like
    if re.match(r"^\d", name):
        name = f"col_{name}"

    return name


def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with column names normalized to snake_case."""
    out = df.copy()
    out.columns = [to_snake_case(col) for col in out.columns]
    return out


def deduplicate_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Append __1, __2, ... suffixes to columns whose name appears more than once."""
    out = df.copy()
    seen = {}
    new_cols = []

    for col in out.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}__{seen[col]}")

    out.columns = new_cols
    return out


# ── Row / value normalization ─────────────────────────────────────────

def remove_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drop exact duplicate rows."""
    return df.drop_duplicates().copy()


def standardize_missing_values(
    df: pd.DataFrame,
    missing_tokens: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """
    Convert semantic missing placeholders (e.g. "n/a", "-", "unknown") to pd.NA.
    Only applied to string-like values; native NaN and non-strings are untouched.
    """
    out = df.copy()
    tokens = {str(x).strip().lower() for x in (missing_tokens or DEFAULT_CLEANING_MISSING_TOKENS)}

    for col in out.columns:
        out[col] = out[col].apply(
            lambda x: pd.NA
            if isinstance(x, str) and x.strip().lower() in tokens
            else x
        )
    return out


def normalize_text_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lowercase and whitespace-normalize textual cells only.
    Keeps nulls and non-text cells unchanged.
    """
    out = df.copy()

    def _norm(x):
        if pd.isna(x):
            return x
        if isinstance(x, str):
            return _collapse_spaces(x).lower()
        return x

    for col in out.columns:
        out[col] = out[col].apply(_norm)

    return out


# ── Cleaning plan loader (reads the SLIM json produced by profiling) ──

def load_cleaning_plan(plan_json_path: str, dataset_key: str) -> dict:
    """
    Load the slim cleaning plan for a single dataset.

    The slim JSON has structure:
      { dataset_key: { col_name: {expected_dtype, dominant_format, transformation_rule} } }

    Returns the per-column cleaning plan for `dataset_key`, or {} if missing.
    """
    with open(plan_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get(dataset_key, {})

In [495]:
# COLUMN ROLES
# Closed vocabulary of roles assignable to columns.
# The LLM picks one of these during profiling. If none fits, it can return
# "custom:<name>" to propose a new role.

COLUMN_ROLES = {
    # numeric
    "identifier_numeric",   # numeric IDs without arithmetic meaning (preserve as string to keep leading zeros)
    "count",                # non-negative integer counts (e.g. number of flights)
    "measure",              # continuous measurements (weight, distance, price)
    "percentage",           # percentage value
    "year",                 # year (e.g. 2024)
    "month_number",         # month as 01-12
    "day_number",           # day as 01-31

    # string / code
    "identifier",           # generic identifier (could be alphanumeric)
    "category",             # categorical variable (e.g. "low", "medium", "high")
    "free_text",            # free text / operator notes
    "flag_binary",          # binary flag (yes/no, 0/1, alto/basso)

    # temporal
    "date",                 # date without time
    "datetime",             # date + time

    # special
    "unknown",              # LLM cannot classify
}

# Deterministic mapping role -> expected pandas dtype.
# The cleaning agent uses this AFTER applying transformations
# to enforce the final column dtype.

ROLE_TO_EXPECTED_DTYPE = {
    "identifier_numeric": "string",      # keep as string to preserve leading zeros
    "count":              "Int64",       # nullable integer
    "measure":            "Float64",     # nullable float
    "percentage":         "Float64",
    "year":               "Int64",
    "month_number":       "Int64",
    "day_number":         "Int64",

    "identifier":         "string",
    "category":           "string",
    "free_text":          "string",
    "flag_binary":        "string",

    "date":               "datetime64[ns]",
    "datetime":           "datetime64[ns]",

    "unknown":            "string",      # safe default
}


def resolve_expected_dtype(role: str) -> str:
    """
    Resolve the expected pandas dtype for a given role.
    Handles 'custom:<name>' roles by defaulting to 'string'.
    """
    if role in ROLE_TO_EXPECTED_DTYPE:
        return ROLE_TO_EXPECTED_DTYPE[role]
    if isinstance(role, str) and role.startswith("custom:"):
        return "string"
    return "string"

In [496]:
# JSON HELPERS
# Utilities to convert pandas/numpy values into JSON-safe native types,
# and to load/write JSON files robustly.

import os
import re
import json
import math
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd


def _to_native(value: Any) -> Any:
    """Convert pandas/numpy objects into native JSON-serializable Python types."""
    if isinstance(value, dict):
        return {str(k): _to_native(v) for k, v in value.items()}
    if isinstance(value, (np.ndarray, list, tuple)):
        return [_to_native(v) for v in value]
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if math.isnan(float(value)):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def _safe_load_json(path: str) -> dict:
    """Load a JSON file safely; return an empty dict if missing/corrupted."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def _safe_write_json(path: str, data: dict) -> None:
    """Write JSON deterministically with UTF-8 and 2-space indentation."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_native(data), f, ensure_ascii=False, indent=2)

In [497]:
# DATAFRAME-LEVEL HELPERS

def get_dataframe_shape(df: pd.DataFrame) -> dict:
    """Return dataframe shape as {rows, columns}."""
    rows, cols = df.shape
    return {"rows": int(rows), "columns": int(cols)}


def get_dataframe_dtypes(df: pd.DataFrame) -> dict:
    """Return pandas dtype per column."""
    return {col: str(dtype) for col, dtype in df.dtypes.items()}


def get_column_missing_stats(df: pd.DataFrame) -> dict:
    result = {}
    for col in df.columns:
        mask = df[col].isna()
        result[col] = {
            "null_count": int(mask.sum()),
            "non_null_count": int((~mask).sum()),
            "missing_percentage": round(float(mask.mean() * 100), 2),
        }
    return result

In [498]:
# =============================================================================
# FORMAT DETECTION
# =============================================================================
# Lightweight format labeling: each value is classified into a coarse
# category (digit, alpha, alphanumeric, ...) used to detect the dominant
# value-format of a column.
#
# We deliberately keep the categories generic. Fine-grained semantic
# inference (is this a country code? a month number? a flight ID?) is
# delegated to the LLM via the role assignment.

def _value_format_label(value: Any) -> str:
    """Return a coarse format label for a single value."""
    if value is None:
        return "missing"

    s = str(value).strip()

    if s == "":
        return "empty"

    # Pure digits (e.g. "5", "12", "00123")
    if s.isdigit():
        return "digit"

    # Float-like (e.g. "12.34", "12,34", "-3.5")
    try:
        float(s.replace(",", "."))
        return "float"
    except Exception:
        pass

    # Pure letters (e.g. "gennaio", "italia")
    if s.isalpha():
        return "alpha"

    # Mixed letters + digits (e.g. "AZ123", "1 voli")
    has_alpha = any(c.isalpha() for c in s)
    has_digit = any(c.isdigit() for c in s)
    if has_alpha and has_digit:
        return "alphanumeric"

    # Anything else (punctuation, symbols, dates with separators, ...)
    return "generic"


def detect_dominant_format(
    series: pd.Series,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    """
    Detect the dominant value-format of a column on a small head() sample.

    Returns:
      {
        "label":       "digit" | "alpha" | "alphanumeric" | "float" | "generic",
        "share_pct":   float,
        "sample_size": int     # how many non-missing rows were actually inspected
      }
    """
    # Take the first `sample_size` rows of the column, then drop missing values
    head_sample = series.head(sample_size)
    clean = head_sample.dropna()

    if clean.empty:
        return {
            "label": None,
            "share_pct": 0.0,
            "sample_size": 0,
        }

    labels = clean.map(_value_format_label)
    counts = labels.value_counts(dropna=False)
    total = int(counts.sum())

    return {
        "label": str(counts.index[0]),
        "share_pct": round(float(counts.iloc[0] / total * 100), 2),
        "sample_size": total,
    }


def collect_non_conforming_samples(
    series: pd.Series,
    dominant_label: str,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
    max_samples: int = 100,
) -> dict:
    """
    Inspect the same head() sample used for dominant format detection
    and collect up to `max_samples` unique values whose format does NOT
    match the dominant label.

    Returns:
      {
        "share_pct": float,        # share of non-conforming values in the sample
        "samples":   list[str],    # up to `max_samples` unique non-conforming values
      }
    """
    head_sample = series.head(sample_size)
    clean = head_sample.dropna()

    if clean.empty or dominant_label is None:
        return {"share_pct": 0.0, "samples": []}

    labels = clean.map(_value_format_label)
    non_conforming_mask = labels != dominant_label
    non_conforming_values = clean.loc[non_conforming_mask]

    total = int(len(clean))
    n_non_conforming = int(non_conforming_mask.sum())

    unique_samples = []
    for v in non_conforming_values.tolist():
        native_v = _to_native(v)
        if native_v not in unique_samples:
            unique_samples.append(native_v)
        if len(unique_samples) >= max_samples:
            break

    return {
        "share_pct": round(float(n_non_conforming / total * 100), 2) if total else 0.0,
        "samples": unique_samples,  
    }


def _try_numeric(series: pd.Series) -> pd.Series:
    """
    Convert a series to numeric where possible (supports comma decimals
    and optional thousands separators). Used only for the numeric_summary
    block in the rich profile.
    """
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    s = series.astype("string").str.strip()
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

In [499]:
# LLM-DRIVEN COLUMN ENRICHMENT
# Given the deterministic part of a column profile (stats + dominant format
# + non-conforming samples), call the LLM ONCE per column to obtain:
#   - role               : one of COLUMN_ROLES (or "custom:<name>")
#   - expected_dtype     : echoed from the role mapping (LLM may override)
#   - transformation_rule: declarative rule the cleaning agent will apply
#   - description        : 1-2 sentence human-readable description

# Closed set of declarative actions the LLM is allowed to choose from.
# The cleaning agent has ONE dispatcher per action; it does NOT need
# column-specific functions like `month_name_to_number`.
TRANSFORMATION_ACTIONS = {
    "none",        # all values already conform
    "extract",     # extract substring matching a pattern (params: regex)
    "map",         # map non-conforming values to canonical ones (params: mapping, canonical_values)
    "set_null",    # explicitly null out broken placeholders
}


def _build_enrichment_prompt(deterministic_profile: dict) -> str:
    return f"""
You are a data profiling assistant.

You are given the deterministic profile of ONE column from a tabular dataset.
Your job is to enrich it with:
  1. a ROLE chosen from a closed vocabulary
  2. a declarative TRANSFORMATION_RULE that explains how to convert
     non-conforming values into the dominant format
  3. a brief DESCRIPTION (1-2 sentences) for human reviewers

==============================================================================
CLOSED VOCABULARY OF ROLES
==============================================================================
{sorted(COLUMN_ROLES)}

If none fits, return a role of the form "custom:<short_snake_case_name>"
and explain why in the description.

==============================================================================
DEFAULT ROLE -> EXPECTED DTYPE MAPPING (for reference)
==============================================================================
{json.dumps(ROLE_TO_EXPECTED_DTYPE, indent=2)}

==============================================================================
CLOSED VOCABULARY OF TRANSFORMATION ACTIONS
==============================================================================
{sorted(TRANSFORMATION_ACTIONS)}

A transformation_rule MUST have the structure:
{{
  "action":        "<one of the actions above>",
  "target_format": "<short description of the expected value format>",
  "params":        {{...}},
  "description":   "<1 sentence explaining what the rule does>"
}}

Examples of params per action:
  - extract  : {{"regex": "\\\\d+"}}
  - map      : {{"mapping": {{"<raw_value_1>": "<canonical_1>",
                               "<raw_value_2>": "<canonical_2>"}},
                 "canonical_values": ["<canonical_1>", "<canonical_2>"]}}
  - set_null : {{}}
  - none     : {{}}

Infer params by looking at the sample values and non-conforming samples
in the column profile. Do NOT assume a fixed format — derive it from the data.

==============================================================================
NUMERIC SUMMARY ANALYSIS
==============================================================================
If the profile contains a "numeric_summary" field, you MUST use it.
The non_conforming samples only capture values whose CHARACTER FORMAT differs
from the dominant format — they CANNOT detect values that are numerically
implausible but correctly formatted (e.g. "24" looks like a valid digit 
just like "2024", so it never appears in non_conforming).

Step 1 — Determine the expected range for this role:
  - year        : 1900-2100
  - month_number: 1-12
  - day_number  : 1-31
  - count       : >= 0
  - percentage  : 0-100
  - measure     : reason from mean and median

Step 2 — Compare min and max against the expected range.
  If min OR max fall outside the expected range, those values are implausible.

Step 3 — Choose the action:
  - If the implausible value appears to be an abbreviated version of a valid
    value (e.g. "24" when the dominant value is "2024"), use "extract" with
    a regex that matches the full valid format.
  - If the implausible value cannot be recovered, use "set_null".
  - NEVER use "map" to fix numerically implausible values.

==============================================================================
COLUMN PROFILE
==============================================================================
{json.dumps(deterministic_profile, ensure_ascii=False, indent=2)}

==============================================================================
OUTPUT
==============================================================================
Return ONLY a JSON object with EXACTLY these keys, no preamble, no markdown:
{{
  "role":                "<role from the vocabulary or custom:...>",
  "transformation_rule": {{...}},
  "description":         "<1-2 sentences>"
}}
""".strip()


def _parse_llm_enrichment_response(response_text: str) -> dict:
    """Parse the LLM JSON response, tolerating ```json fences and extra whitespace."""
    text = response_text.strip()
    # Strip optional markdown fences
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def _fallback_enrichment(deterministic_profile: dict, error: str) -> dict:
    """Used when the LLM call fails or returns invalid JSON."""
    return {
        "role": "unknown",
        "role_source": "fallback",
        "expected_dtype": "string",
        "transformation_rule": {
            "action": "none",
            "target_format": deterministic_profile.get("dominant_format", {}).get("label"),
            "params": {},
            "description": "Fallback: no transformation applied (LLM enrichment failed).",
        },
        "description": f"LLM enrichment failed: {error}",
    }


def _validate_and_fix_enrichment(
    enrichment: dict,
    deterministic_profile: dict,
) -> dict:
    fixes = []

    role = enrichment.get("role", "unknown")
    expected_dtype = enrichment.get("expected_dtype")
    rule = enrichment.get("transformation_rule") or {}
    action = rule.get("action", "none")
    params = rule.get("params") or {}

    dominant = (deterministic_profile.get("dominant_format") or {}).get("label")
    dominant_share = (deterministic_profile.get("dominant_format") or {}).get("share_pct", 0)

    # ── FIX 1: formato dominante testuale ma role numerico ────────────
    NUMERIC_ROLES = {
        "identifier_numeric", "count", "measure", "percentage",
        "year", "month_number", "day_number",
    }
    if dominant in {"alpha", "generic"} and dominant_share >= 80 and role in NUMERIC_ROLES:
        n_unique = deterministic_profile.get("n_unique_non_null", 0)
        row_count = deterministic_profile.get("row_count", 1)
        unique_ratio = n_unique / max(row_count, 1)

        new_role = "category" if unique_ratio < 0.05 else "free_text"
        new_dtype = resolve_expected_dtype(new_role)

        fixes.append(
            f"role '{role}' is numeric but dominant_format is '{dominant}' at "
            f"{dominant_share}% (textual evidence overrides). "
            f"Reassigned to '{new_role}', dtype to '{new_dtype}'."
        )
        role = new_role
        expected_dtype = new_dtype
        action = "none"
        params = {}

    # ── FIX 2: transformation_rule sempre completa ────────────────────
    fixed_rule = {
        "action": action,
        "target_format": rule.get("target_format", dominant),
        "params": params,
        "description": rule.get("description", ""),
    }

    out = dict(enrichment)
    out["role"] = role
    out["expected_dtype"] = expected_dtype
    out["transformation_rule"] = fixed_rule
    if fixes:
        out["validator_fixes"] = fixes
    return out


def enrich_column_profile_with_llm(
    deterministic_profile: dict,
    llm_callable: Optional[Callable[[str], str]] = None,
) -> dict:
    if llm_callable is None:
        return _fallback_enrichment(deterministic_profile, "no llm_callable provided")

    prompt = _build_enrichment_prompt(deterministic_profile)

    try:
        raw = llm_callable(prompt)
        parsed = _parse_llm_enrichment_response(str(raw))
    except Exception as e:
        return _fallback_enrichment(deterministic_profile, str(e))

    # Validate role
    role = parsed.get("role", "unknown")
    if role in COLUMN_ROLES:
        role_source = "llm"
    elif isinstance(role, str) and role.startswith("custom:"):
        role_source = "custom"
    else:
        role = "unknown"
        role_source = "fallback"

    # expected_dtype sempre da ROLE_TO_EXPECTED_DTYPE
    expected_dtype = resolve_expected_dtype(role)

    # Validate transformation_rule
    rule = parsed.get("transformation_rule") or {}
    if not isinstance(rule, dict) or rule.get("action") not in TRANSFORMATION_ACTIONS:
        rule = {
            "action": "none",
            "target_format": deterministic_profile.get("dominant_format", {}).get("label"),
            "params": {},
            "description": "LLM returned an invalid transformation_rule; defaulted to 'none'.",
        }
    rule.setdefault("params", {})
    rule.setdefault("target_format", deterministic_profile.get("dominant_format", {}).get("label"))
    rule.setdefault("description", "")

    enrichment = {
        "role": role,
        "role_source": role_source,
        "expected_dtype": expected_dtype,
        "transformation_rule": rule,
        "description": str(parsed.get("description", "")).strip(),
    }

    enrichment = _validate_and_fix_enrichment(enrichment, deterministic_profile)
    return enrichment

In [500]:
# COLUMN PROFILING — orchestration
# Builds the rich per-column profile (deterministic stats + LLM enrichment)
# and then derives the slim cleaning plan that will be fed to the cleaning agent.

def profile_single_column(
    df: pd.DataFrame,
    column_name: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    series = df[column_name]
    non_missing = series.dropna()

    sample_values = [_to_native(v) for v in non_missing.head(5).tolist()]

    dominant = detect_dominant_format(series, sample_size=sample_size)
    non_conforming = collect_non_conforming_samples(
        series,
        dominant_label=dominant["label"],
        sample_size=sample_size,
        max_samples=100,
    )

    missing_stats = get_column_missing_stats(df[[column_name]])[column_name]

    deterministic_profile = {
        "column_name": column_name,
        "dtype": str(series.dtype),
        "row_count": int(len(series)),
        "null_count": missing_stats["null_count"],
        "non_null_count": missing_stats["non_null_count"],
        "missing_percentage": missing_stats["missing_percentage"],
        "n_unique_non_null": int(non_missing.nunique(dropna=True)),
        "unique_ratio_non_null": round(
            float(non_missing.nunique(dropna=True) / max(len(non_missing), 1)), 4
        ),
        "sample_values": sample_values,
        "dominant_format": dominant,
        "non_conforming": non_conforming,
    }

    numeric = _try_numeric(non_missing)
    if len(non_missing) > 0 and numeric.notna().mean() >= 0.9:
        numeric = numeric.dropna()
        if not numeric.empty:
            deterministic_profile["numeric_summary"] = {
                "min": _to_native(numeric.min()),
                "max": _to_native(numeric.max()),
                "mean": _to_native(round(float(numeric.mean()), 4)),
                "median": _to_native(round(float(numeric.median()), 4)),
                "std": _to_native(round(float(numeric.std(ddof=1)), 4)) if len(numeric) > 1 else None,
            }

    enrichment = enrich_column_profile_with_llm(
        deterministic_profile=deterministic_profile,
        llm_callable=llm_callable,
    )

    full_profile = {**deterministic_profile, **enrichment}
    return _to_native(full_profile)


def profile_all_columns(
    df: pd.DataFrame,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    """Build the rich profile for every column of a dataframe."""
    return {
        col: profile_single_column(df, col, llm_callable=llm_callable, sample_size=sample_size)
        for col in df.columns
    }


# Slim cleaning plan
# Distilled view of the rich profile. Contains ONLY what the cleaning agent
# needs to operate. Fed to the cleaning step as cleaning_plan.json.

def build_cleaning_plan(rich_profile_per_column: dict) -> dict:
    """
    Reduce the rich per-column profile into the slim cleaning plan.
    Input  : {col_name: rich_profile_dict, ...}
    Output : {col_name: {expected_dtype, dominant_format, transformation_rule}}
    """
    plan = {}
    for col_name, profile in rich_profile_per_column.items():
        dominant = profile.get("dominant_format", {}) or {}
        plan[col_name] = {
            "column_name": col_name,
            "expected_dtype": profile.get("expected_dtype", "string"),
            "dominant_format": dominant.get("label"),
            "transformation_rule": profile.get("transformation_rule", {
                "action": "none",
                "target_format": dominant.get("label"),
                "params": {},
                "description": "",
            }),
        }
    return plan


# Main orchestration — writes BOTH JSON files

def build_dataset_column_profiles(
    df: pd.DataFrame,
    dataset_name: str,
    rich_json_path: str,
    slim_json_path: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
    findings_root_key: str = "column_profiles",
) -> dict:
    """
    Build the rich JSON profile for every column and ALSO write the slim
    cleaning plan derived from it.

    Rich JSON structure:
      {
        findings_root_key: {
          dataset_name: {
            "dataset_summary": {...},
            "columns": {col_name: rich_profile, ...}
          }
        }
      }

    Slim JSON structure:
      {
        dataset_name: {col_name: {expected_dtype, dominant_format, transformation_rule}}
      }
    """
    # ── Build rich profile 
    findings = _safe_load_json(rich_json_path)

    dataset_summary = {
        "dataset_name": dataset_name,
        "shape": get_dataframe_shape(df),
        "dtypes": get_dataframe_dtypes(df),
    }

    columns_payload = profile_all_columns(
        df,
        llm_callable=llm_callable,
        sample_size=sample_size,
    )

    findings.setdefault(findings_root_key, {})
    findings[findings_root_key][dataset_name] = {
        "dataset_summary": dataset_summary,
        "columns": columns_payload,
    }
    _safe_write_json(rich_json_path, findings)

    #  Build slim cleaning plan 
    slim_all = _safe_load_json(slim_json_path)
    slim_all[dataset_name] = build_cleaning_plan(columns_payload)
    _safe_write_json(slim_json_path, slim_all)

    return findings[findings_root_key][dataset_name]

In [501]:
# =============================================================================
# CLEANING RULE APPLICATION (replaces previous _row_matches_format gate)
# =============================================================================
# Each action computes its OWN target mask, coherent with its semantics.
# - map      : target = rows whose normalized key is in the mapping
# - extract  : target = rows whose value does NOT already fullmatch the regex
# - set_null : "smart" — try to recover the value before nulling
# - none     : identity
import re

# helpers

def _is_missing(v) -> bool:
    return v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA


def _smart_recover(value, dominant_label: Optional[str]):
    """
    For set_null: try to recover a canonical value coherent with dominant_label.
    Returns the recovered value, or pd.NA if not recoverable.
    """
    if _is_missing(value):
        return value

    s = str(value).strip()
    if s == "":
        return pd.NA

    if dominant_label == "digit":
        # Negative-signed values are intentionally dropped (LLM plan intent)
        if "-" in s:
            return pd.NA
        m = re.search(r"\d+", s)
        return m.group(0) if m else pd.NA

    if dominant_label == "float":
        m = re.search(r"-?\d+[.,]?\d*", s)
        if not m:
            return pd.NA
        return m.group(0).replace(",", ".")

    if dominant_label == "alpha":
        m = re.search(r"[A-Za-z]+", s)
        return m.group(0) if m else pd.NA

    if dominant_label == "alphanumeric":
        m = re.search(r"[A-Za-z0-9]+", s)
        return m.group(0) if m else pd.NA

    # "generic" or None: nothing reliable to extract
    return pd.NA


# per-action target masks 

def _mask_for_map(series: pd.Series, params: dict) -> pd.Series:
    mapping = params.get("mapping", {}) or {}
    if not mapping:
        return pd.Series(False, index=series.index)
    keys = {str(k).strip().lower() for k in mapping.keys()}

    def _hit(v):
        if _is_missing(v):
            return False
        return str(v).strip().lower() in keys

    return series.map(_hit).astype(bool)


def _mask_for_extract(series: pd.Series, params: dict) -> pd.Series:
    pattern = params.get("regex", r"\d+")
    full = re.compile(f"^(?:{pattern})$")

    def _needs_extract(v):
        if _is_missing(v):
            return False
        return full.match(str(v).strip()) is None

    return series.map(_needs_extract).astype(bool)


def _mask_for_set_null(series: pd.Series, dominant_label: Optional[str]) -> pd.Series:
    """Non-conforming = label != dominant_label (using _value_format_label)."""
    if dominant_label is None:
        return series.notna()  # without dominant we treat all non-null as targets

    def _non_conforming(v):
        if _is_missing(v):
            return False
        return _value_format_label(v) != dominant_label

    return series.map(_non_conforming).astype(bool)


# action implementations 

def _action_none(series: pd.Series, params: dict) -> pd.Series:
    return series


def _action_extract(series: pd.Series, params: dict) -> pd.Series:
    pattern = params.get("regex", r"\d+")
    return series.astype("string").str.extract(f"({pattern})", expand=False)


def _action_map(series: pd.Series, params: dict) -> pd.Series:
    mapping = params.get("mapping", {}) or {}
    if not mapping:
        return series
    norm = {str(k).strip().lower(): (pd.NA if v is None else str(v))
            for k, v in mapping.items()}

    def _map_one(v):
        if _is_missing(v):
            return v
        return norm.get(str(v).strip().lower(), v)

    return series.map(_map_one)


def _action_set_null(series: pd.Series, params: dict, dominant_label: Optional[str]) -> pd.Series:
    """Smart set_null: try to recover; if not recoverable → pd.NA."""
    return series.map(lambda v: _smart_recover(v, dominant_label))


ACTION_DISPATCHER = {
    "none":     _action_none,
    "extract":  _action_extract,
    "map":      _action_map,
    "set_null": _action_set_null,  # signature handled specially below
}


# orchestration 

def apply_transformation_rule(
    series: pd.Series,
    transformation_rule: dict,
    dominant_label: Optional[str],
) -> pd.Series:
    action = (transformation_rule or {}).get("action", "none")
    params = (transformation_rule or {}).get("params", {}) or {}

    if action == "none" or action not in ACTION_DISPATCHER:
        return series

    # Per-action target mask
    if action == "map":
        target_mask = _mask_for_map(series, params)
    elif action == "extract":
        target_mask = _mask_for_extract(series, params)
    elif action == "set_null":
        target_mask = _mask_for_set_null(series, dominant_label)
    else:
        target_mask = pd.Series(False, index=series.index)

    if not target_mask.any():
        return series

    out = series.astype("object").copy()
    sub = out.loc[target_mask]

    if action == "set_null":
        out.loc[target_mask] = _action_set_null(sub, params, dominant_label)
    else:
        out.loc[target_mask] = ACTION_DISPATCHER[action](sub, params)

    return out


#  dtype enforcement (unchanged) 

def enforce_expected_dtype(series: pd.Series, expected_dtype: str) -> pd.Series:
    if expected_dtype is None or str(series.dtype) == expected_dtype:
        return series

    try:
        if expected_dtype.startswith("datetime"):
            return pd.to_datetime(series, errors="coerce")
        if expected_dtype in {"Int64", "Int32"}:
            return pd.to_numeric(series, errors="coerce").astype(expected_dtype)
        if expected_dtype in {"Float64", "Float32", "float64", "float32"}:
            return pd.to_numeric(series, errors="coerce").astype(expected_dtype)
        return series.astype(expected_dtype)
    except Exception:
        return series

# clean_dataset

def clean_dataset(
    input_csv_path: str,
    output_csv_path: str,
    cleaning_plan_path: str,
    dataset_key: str,
    read_csv_kwargs: Optional[dict] = None,
) -> dict:
    read_csv_kwargs = read_csv_kwargs or {}

    df = pd.read_csv(input_csv_path, **read_csv_kwargs)
    shape_before = df.shape

    df = normalize_column_names(df)
    df = deduplicate_column_names(df)
    df = standardize_missing_values(df)
    df = normalize_text_values(df)

    plan = load_cleaning_plan(cleaning_plan_path, dataset_key)

    per_column_actions = {}
    dtype_changes = {}

    for col in df.columns:
        col_plan = plan.get(col)
        if not col_plan:
            per_column_actions[col] = {"action": "skipped (no plan)"}
            continue

        rule = col_plan.get("transformation_rule", {}) or {}
        dominant_label = col_plan.get("dominant_format")
        expected_dtype = col_plan.get("expected_dtype")

        original = df[col]
        df[col] = apply_transformation_rule(
            series=df[col],
            transformation_rule=rule,
            dominant_label=dominant_label,
        )

        dtype_before = str(df[col].dtype)
        df[col] = enforce_expected_dtype(df[col], expected_dtype)
        dtype_after = str(df[col].dtype)

        n_changed = int((original.astype("string") != df[col].astype("string")).sum())
        per_column_actions[col] = {
            "action": rule.get("action", "none"),
            "rows_changed": n_changed,
            "dtype_before": dtype_before,
            "dtype_after": dtype_after,
            "expected_dtype": expected_dtype,
            "dtype_match": (dtype_after == expected_dtype) if expected_dtype else None,
        }
        if dtype_before != dtype_after:
            dtype_changes[col] = {"before": dtype_before, "after": dtype_after}

    df = remove_duplicate_rows(df)

    # ── DTYPE DIAGNOSTIC — stampa lo stato dei dtype prima di scrivere il CSV
    print(f"\n{'='*70}")
    print(f"DTYPE DIAGNOSTIC for {dataset_key}  (pre-to_csv)")
    print('='*70)
    print(f"{'column':<30} {'expected':<20} {'actual':<20} OK?")
    print('-'*70)
    for col, info in per_column_actions.items():
        if info.get('action') == 'skipped (no plan)':
            continue
        exp = info.get('expected_dtype') or '(none)'
        act = info.get('dtype_after') or '?'
        ok = '✅' if (info.get('expected_dtype') is None or info.get('dtype_match')) else '❌'
        print(f"{col:<30} {exp:<20} {act:<20} {ok}")
    print('='*70 + '\n')

    shape_after = df.shape
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    return {
        "dataset_key": dataset_key,
        "input_path": input_csv_path,
        "output_path": output_csv_path,
        "shape_before": list(shape_before),
        "shape_after": list(shape_after),
        "duplicate_rows_removed": int(shape_before[0] - shape_after[0]),
        "per_column_actions": per_column_actions,
        "dtype_changes": dtype_changes,
    }

In [502]:
def llm_callable(prompt: str) -> str:
    response = _client.chat.completions.create(
        model=MISTRAL_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data profiling assistant. "
                    "When asked, you return ONLY a valid JSON object — "
                    "no preamble, no markdown fences, no explanations outside the JSON. "
                    "Do not invent facts not supported by the provided profile."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=1024,
        response_format={"type": "json_object"},
    )
    return response.choices[0].message.content.strip()

def run_pre_pipeline_column_profiling(
    datasets: list[tuple[str, str]],
) -> dict:
    profiles = {}

    for dataset_name, csv_path in datasets:
        if not os.path.exists(csv_path):
            print(f"  [profiling] File not found, skipped: {csv_path}")
            continue

        try:
            original_cols = list(pd.read_csv(csv_path, nrows=0).columns)
            print(f"  [profiling] Running column profiling for {dataset_name} "
                  f"({len(original_cols)} columns: {', '.join(original_cols[:5])}"
                  f"{'...' if len(original_cols) > 5 else ''})")
        except Exception:
            print(f"  [profiling] Running column profiling for {dataset_name}...")

        df = pd.read_csv(csv_path)
        df = normalize_column_names(df)
        df = deduplicate_column_names(df)
        df = standardize_missing_values(df)
        df = normalize_text_values(df)

        profiles[dataset_name] = build_dataset_column_profiles(
            df=df,
            dataset_name=dataset_name,
            rich_json_path=COLUMN_PROFILES_JSON,
            slim_json_path=CLEANING_PLAN_JSON,
            llm_callable=llm_callable,
            sample_size=DOMINANT_FORMAT_SAMPLE_SIZE,
        )
        print(f"  [profiling] Saved profile for {dataset_name}")

    return profiles


# Writing json files

In [503]:
print("=" * 60)
print("  PRE-PIPELINE COLUMN PROFILING")
print("=" * 60)

if os.path.exists(COLUMN_PROFILES_JSON) and os.path.exists(CLEANING_PLAN_JSON):
    print(f"  [cache] Profiles already exist, skipping profiling.")
    print(f"  ✓ {COLUMN_PROFILES_JSON} ({os.path.getsize(COLUMN_PROFILES_JSON):,} bytes)")
    print(f"  ✓ {CLEANING_PLAN_JSON} ({os.path.getsize(CLEANING_PLAN_JSON):,} bytes)")
else:
    profiles = run_pre_pipeline_column_profiling(
        datasets=[
            ("allarmi_raw",    ALLARMI_CSV),
            ("tipologia_raw",  TIPOLOGIA_CSV),
        ]
    )

    print()
    print("Files written:")
    if os.path.exists(COLUMN_PROFILES_JSON):
        print(f"  ✓ {COLUMN_PROFILES_JSON} ({os.path.getsize(COLUMN_PROFILES_JSON):,} bytes)")
    else:
        print(f"  ✗ {COLUMN_PROFILES_JSON} (NOT created)")

    if os.path.exists(CLEANING_PLAN_JSON):
        print(f"  ✓ {CLEANING_PLAN_JSON} ({os.path.getsize(CLEANING_PLAN_JSON):,} bytes)")
    else:
        print(f"  ✗ {CLEANING_PLAN_JSON} (NOT created)")

  PRE-PIPELINE COLUMN PROFILING
  [cache] Profiles already exist, skipping profiling.
  ✓ /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/column_profiles.json (88,484 bytes)
  ✓ /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/cleaning_plan.json (28,087 bytes)


# Cleaning Agent

In [504]:
print("=" * 60)
print("  CLEANING AGENT")
print("=" * 60)

datasets_to_clean = [
    ("allarmi_raw",   ALLARMI_CSV,   os.path.join(OUTPUT_DIR, "allarmi_clean.csv")),
    ("tipologia_raw", TIPOLOGIA_CSV, os.path.join(OUTPUT_DIR, "tipologia_clean.csv")),
]

for dataset_key, input_path, output_path in datasets_to_clean:
    if not os.path.exists(input_path):
        print(f"  [cleaning] File not found, skipped: {input_path}")
        continue

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f"  [cache] '{dataset_key}' already cleaned → {os.path.basename(output_path)} ({os.path.getsize(output_path):,} bytes), skipping.")
        continue

    print(f"  [cleaning] Processing '{dataset_key}'...")
    report = clean_dataset(
        input_csv_path=input_path,
        output_csv_path=output_path,
        cleaning_plan_path=CLEANING_PLAN_JSON,
        dataset_key=dataset_key,
    )

    print(f"  [cleaning] Shape: {report['shape_before']} → {report['shape_after']}")
    print(f"  [cleaning] Duplicate rows removed: {report['duplicate_rows_removed']}")
    print(f"  [cleaning] Dtype changes: {report['dtype_changes']}")
    print(f"  ✓ Saved: {output_path}")
    print()

  CLEANING AGENT
  [cache] 'allarmi_raw' already cleaned → allarmi_clean.csv (861,879 bytes), skipping.
  [cache] 'tipologia_raw' already cleaned → tipologia_clean.csv (1,102,987 bytes), skipping.


In [505]:
# ── FINDINGS HELPER ──────────────────────────────────────────────────────────
# Returns a snippet of prompt instructions telling the agent how to read/write
# the shared findings JSON. Each agent updates ITS OWN key, preserving others.

def _findings_guidance(task_key: str, extra_notes: str = "") -> str:
    base = (
        f"Maintain a shared findings JSON at '{FINDINGS_JSON}'. "
        f"At the start, attempt to load it; if missing, empty, invalid, or corrupted, "
        f"initialize an empty dict instead of failing. "
        f"Store new information under the key '{task_key}' while preserving existing keys "
        f"for other tasks. "
        f"Use concise, machine-readable fields with only native Python JSON-serializable types "
        f"such as dict, list, str, int, float, bool, or null. "
        f"Convert pandas and numpy values to native Python types before saving. "
        f"Convert tuples to lists before saving. "
        f"After completing the task, update the entry for '{task_key}' and write the full JSON "
        f"back by overwriting the file. "
    )
    if extra_notes:
        base += extra_notes
    return base


print("✓ _findings_guidance ready")

✓ _findings_guidance ready


# Executor

In [506]:
# ── CODE EXECUTOR (v2: robust extractor + larger max_tokens) ─────────────────
# Sends a prompt to Mistral, gets Python code back, executes it in a REPL,
# captures stdout/stderr. No judgment, just execution.

import io, sys, traceback, re

CODE_EXECUTOR_SYSTEM_PROMPT = (
    "You are a code-generation agent. "
    "When given a task, you respond ONLY with executable Python code — "
    "no preamble, no explanations, no markdown fences. "
    "Use only standard libraries plus pandas, numpy, json, os, re, math. "
    "Do not import subprocess, requests, or any networking module. "
    "All inputs and outputs are file paths on the local filesystem. "
    "Always print short progress messages so the orchestrator can see what happened. "
    "Always end with a clear print line confirming success or raising an explicit error."
)


def _extract_code_from_response(raw: str) -> str:
    """
    Strip markdown fences if Mistral wraps the code despite instructions.
    Handles three cases:
      1. Bare code (no fences)
      2. Fenced and properly closed:  ```python ... ```
      3. Fenced but truncated (no closing fence) — common when max_tokens is hit
    """
    s = raw.strip()

    # Case 2: matched opening + closing fences
    m = re.search(r"```(?:python)?\s*(.*?)```", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    # Case 3: opening fence with no closing (truncated response)
    m = re.search(r"```(?:python)?\s*\n(.*)", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    # Case 1: bare code, no fences at all
    return s


def code_executor_run(
    prompt: str,
    temperature: float = 0.0,
    max_tokens: int = 4096,        # ← raised from 2048 to avoid truncation
) -> dict:
    """
    One round-trip: prompt → Mistral → code → exec → captured output.
    Returns:
      {
        "code":    str,   # the code Mistral generated
        "stdout":  str,   # everything the code printed
        "stderr":  str,   # captured stderr (incl. tracebacks if exec failed)
        "success": bool,  # True if exec did not raise
      }
    """
    response = _client.chat.completions.create(
        model=MISTRAL_MODEL,
        messages=[
            {"role": "system", "content": CODE_EXECUTOR_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    raw = response.choices[0].message.content
    code = _extract_code_from_response(raw)

    captured_stdout = io.StringIO()
    captured_stderr = io.StringIO()
    success = True

    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = captured_stdout, captured_stderr

    try:
        exec(code, {"__name__": "__agent__"})
    except Exception:
        success = False
        captured_stderr.write(traceback.format_exc())
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr

    return {
        "code":    code,
        "stdout":  captured_stdout.getvalue(),
        "stderr":  captured_stderr.getvalue(),
        "success": success,
    }


print("✓ Code Executor v2 ready (extractor robust + max_tokens=4096)")

✓ Code Executor v2 ready (extractor robust + max_tokens=4096)


# Validator

In [507]:
# ── VALIDATOR ────────────────────────────────────────────────────────────────
# Deterministic checks on the artifact produced by the Baseline Agent.
# No LLM, pure pandas inspection.

def validate_baseline_output(output_csv_path: str) -> dict:
    """
    Validates the dataframe produced by the Baseline Agent.
    """

    import os
    import pandas as pd
    import numpy as np

    checks = []

    # ── 1) File exists ─────────────────────────────
    file_exists = os.path.exists(output_csv_path)
    checks.append({
        "name":   "file_exists",
        "ok":     file_exists,
        "detail": output_csv_path,
    })
    if not file_exists:
        return {"passed": False, "checks": checks}

    # ── 2) Loadable ────────────────────────────────
    try:
        df = pd.read_csv(output_csv_path)
    except Exception as e:
        checks.append({"name": "loadable", "ok": False, "detail": str(e)})
        return {"passed": False, "checks": checks}

    checks.append({"name": "loadable", "ok": True, "detail": f"shape={df.shape}"})

    # ── 3) Required columns (UPDATED) ─────────────
    required = [
        "route",
        "rolling_mean_alarms",
        "rolling_std_alarms",
        "total_alarms",
        "n_records",
        "alert_rate",
    ]

    missing = [c for c in required if c not in df.columns]

    checks.append({
        "name":   "required_columns",
        "ok":     len(missing) == 0,
        "detail": f"missing={missing}" if missing else f"all present: {required}",
    })

    # ── 4) Non-empty ──────────────────────────────
    checks.append({
        "name":   "non_empty",
        "ok":     len(df) > 0,
        "detail": f"rows={len(df)}",
    })

    # ── 5) Numeric checks ─────────────────────────
    if len(missing) == 0:

        numeric_cols = [
            "rolling_mean_alarms",
            "rolling_std_alarms",
            "total_alarms",
            "n_records",
            "alert_rate",
        ]

        non_numeric = [
            c for c in numeric_cols
            if not pd.api.types.is_numeric_dtype(df[c])
        ]

        checks.append({
            "name":   "numeric_dtypes",
            "ok":     len(non_numeric) == 0,
            "detail": f"non_numeric={non_numeric}" if non_numeric else "all numeric",
        })

        # ── 6) No infinite values ─────────────────
        inf_cols = [
            c for c in numeric_cols
            if np.isinf(df[c].fillna(0)).any()
        ]

        checks.append({
            "name":   "no_infinity",
            "ok":     len(inf_cols) == 0,
            "detail": f"inf_in={inf_cols}" if inf_cols else "no inf",
        })

        # ── 7) route populated ───────────────────
        route_ok = (
            df["route"].notna().all()
            and (df["route"].astype(str).str.strip() != "").all()
        )

        checks.append({
            "name":   "route_populated",
            "ok":     route_ok,
            "detail": "all rows have a route" if route_ok else "some routes are empty",
        })

    passed = all(c["ok"] for c in checks)
    return {"passed": passed, "checks": checks}


def print_validation_report(verdict: dict, label: str = ""):
    print(f"\n  ┌─ Validation report {label} ─")
    for c in verdict["checks"]:
        icon = "✓" if c["ok"] else "✗"
        print(f"  │ {icon}  {c['name']:<22} {c['detail']}")
    print(f"  └─ Overall: {'PASSED' if verdict['passed'] else 'FAILED'}\n")


print("✓ Baseline validator ready")


def validate_outlier_output(output_csv_path: str) -> dict:
    checks = []

    file_exists = os.path.exists(output_csv_path)
    checks.append({"name": "file_exists", "ok": file_exists, "detail": output_csv_path})
    if not file_exists:
        return {"passed": False, "checks": checks}

    try:
        df = pd.read_csv(output_csv_path)
    except Exception as e:
        checks.append({"name": "loadable", "ok": False, "detail": str(e)})
        return {"passed": False, "checks": checks}

    checks.append({"name": "loadable", "ok": True, "detail": f"shape={df.shape}"})

    required = ["route", "anomaly_score", "is_outlier"]
    missing = [c for c in required if c not in df.columns]

    checks.append({
        "name": "required_columns",
        "ok": len(missing) == 0,
        "detail": f"missing={missing}" if missing else f"all present: {required}",
    })

    checks.append({
        "name": "non_empty",
        "ok": len(df) > 0,
        "detail": f"rows={len(df)}",
    })

    if "is_outlier" in df.columns:
        checks.append({
            "name": "outliers_flagged",
            "ok": df["is_outlier"].astype(bool).sum() > 0,
            "detail": f"outliers={df['is_outlier'].astype(bool).sum()}",
        })

    passed = all(c["ok"] for c in checks)
    return {"passed": passed, "checks": checks}

✓ Baseline validator ready


# Supervisor

In [508]:
# ── SUPERVISOR ───────────────────────────────────────────────────────────────
# Deterministic orchestration of executor + validator with retry-with-temperature.
# Strategy: first attempt at temperature=0.0; on failure, retry with progressively
# higher temperature, same prompt (no LLM-driven feedback).

def run_agent_with_supervisor(
    task_name: str,
    prompt: str,
    validator_fn,                # callable(output_path) -> {"passed": bool, "checks": [...]}
    output_path: str,
    max_retries: int = 3,
    retry_temperatures: tuple = (0.0, 0.3, 0.6, 0.9),
) -> dict:
    """
    Runs an agent task: executes the code, validates the output, retries if needed.
    Returns:
      {
        "task":       str,
        "succeeded":  bool,
        "attempts":   int,
        "last_code":  str,
        "last_stdout": str,
        "last_stderr": str,
        "last_validation": dict,
      }
    """
    print(f"\n{'='*70}")
    print(f"  AGENT: {task_name}")
    print('='*70)

    last_exec = None
    last_verdict = None

    for attempt_idx in range(max_retries + 1):
        temp = retry_temperatures[min(attempt_idx, len(retry_temperatures) - 1)]
        print(f"\n  [attempt {attempt_idx + 1}/{max_retries + 1}] temperature={temp}")

        # Execute
        last_exec = code_executor_run(prompt, temperature=temp)

        # Echo the code stdout (helps debugging in notebook)
        if last_exec["stdout"]:
            print("  ── stdout ──")
            for line in last_exec["stdout"].rstrip().splitlines():
                print(f"  │ {line}")

        # If exec crashed, surface the traceback
        if not last_exec["success"]:
            print("  ── EXEC FAILED ──")
            tail = last_exec["stderr"].rstrip().splitlines()[-8:]
            for line in tail:
                print(f"  ! {line}")
            continue

        # Validate the produced artifact
        last_verdict = validator_fn(output_path)
        print_validation_report(last_verdict, label=f"attempt {attempt_idx + 1}")

        if last_verdict["passed"]:
            print(f"  ✓ {task_name} SUCCEEDED on attempt {attempt_idx + 1}")
            return {
                "task":       task_name,
                "succeeded":  True,
                "attempts":   attempt_idx + 1,
                "last_code":  last_exec["code"],
                "last_stdout": last_exec["stdout"],
                "last_stderr": last_exec["stderr"],
                "last_validation": last_verdict,
            }

    print(f"\n  ✗ {task_name} FAILED after {max_retries + 1} attempts")
    return {
        "task":       task_name,
        "succeeded":  False,
        "attempts":   max_retries + 1,
        "last_code":  last_exec["code"] if last_exec else "",
        "last_stdout": last_exec["stdout"] if last_exec else "",
        "last_stderr": last_exec["stderr"] if last_exec else "",
        "last_validation": last_verdict,
    }


print("✓ Supervisor ready")

✓ Supervisor ready


# Baseline Agent

In [509]:
# ── Baseline Agent prompt (clean separation) ─────────────────────────────
def _build_baseline_prompt():
    return (
        "You are the Baseline Agent in a multi-agent anomaly detection pipeline. "
        "Your task is to build a route-level baseline dataset. "

        f"Inputs: "
        f"- '{OUTPUT_DIR}/tipologia_clean.csv' "
        f"- '{OUTPUT_DIR}/allarmi_clean.csv' "

        f"Output: '{OUTPUT_DIR}/baseline_data.csv'. "

        "Generate Python code using pandas, numpy, json, os. "

        "STEP 1 — LOAD DATA: "
        "Load both CSVs and print shapes. "

        "STEP 2 — COLUMN DISCOVERY: "
        "Identify departure_airport, arrival_airport. "
        "airport words = ['areoporto','aeroporto','airport','porto'] "
        "departure words = ['partenza','depart','origin'] "
        "arrival words = ['arrivo','arrival','destinaz'] "

        "STEP 3 — SELECT DATASET: "
        "Evaluate each dataset as a candidate. "

        "For each dataset: "
        "- Check if departure and arrival columns are found. "
        "- Check if at least one numeric column can be used as a measure. "

        "A dataset is valid only if BOTH conditions are satisfied. "

        "If multiple datasets are valid, choose the one with the largest number of rows. "

        "If no dataset satisfies both conditions, raise ValueError. "

        "STEP 4 — MEASURE: "
        "Select numeric column. Prefer 'allarmati'. "
        "Create df['_measure'] as numeric. "

        "STEP 5 — BUILD ROUTE: "
        "df['route'] = df[departure].astype(str) + '→' + df[arrival].astype(str) "

        "STEP 6 — AGGREGATE BY ROUTE: "
        "baseline_data = df.groupby('route', dropna=False).agg("
        "rolling_mean_alarms=('_measure','mean'), "
        "rolling_std_alarms=('_measure','std'), "
        "total_alarms=('_measure','sum'), "
        "n_records=('_measure','count')"
        ").reset_index() "

        "STEP 7 — FEATURE ENGINEERING: "
        "baseline_data['alert_rate'] = baseline_data['total_alarms'] / baseline_data['n_records'] "

        "STEP 8 — OUTPUT: "
        "Write baseline_data.csv. "

        "IMPORTANT: "
        "Do NOT compute anomalies. "
        "Do NOT compute z-score. "
        "Do NOT create any outlier flag. "
    )

In [510]:
# ── RUN: Baseline Agent ──────────────────────────────────────────────────────
baseline_output_csv = os.path.join(OUTPUT_DIR, "baseline_data.csv")

result = run_agent_with_supervisor(
    task_name="baseline",
    prompt=_build_baseline_prompt(),
    validator_fn=validate_baseline_output,
    output_path=baseline_output_csv,
    max_retries=3,
)

# Quick peek at the produced dataframe (if successful)
if result["succeeded"]:
    df_baseline = pd.read_csv(baseline_output_csv)
    print(f"\nbaseline_data.csv: shape={df_baseline.shape}")
    print(df_baseline.head())


  AGENT: baseline

  [attempt 1/4] temperature=0.0
  ── stdout ──
  │ Loading datasets...
  │ tipologia_clean.csv shape: (5034, 33)
  │ allarmi_clean.csv shape: (5028, 24)
  │ tipologia departure: areoporto_partenza, arrival: areoporto_arrivo, airport: areoporto_arrivo
  │ allarmi departure: areoporto_partenza, arrival: areoporto_arrivo, airport: areoporto_arrivo
  │ Selected dataset: tipologia with 5034 rows
  │ Departure column: areoporto_partenza, Arrival column: areoporto_arrivo
  │ Selected measure column: allarmati
  │ Writing baseline_data.csv to /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/baseline_data.csv...
  │ SUCCESS: baseline_data.csv generated successfully.

  ┌─ Validation report attempt 1 ─
  │ ✓  file_exists            /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/baseline_data.csv
  │ ✓  loadable               shape=(467, 6)
  │ ✓  required_columns       all present: [

# Outlier Agent

In [511]:
# ── Outlier Detection Agent prompt (clean separation) ─────────────────────
def _build_outlier_prompt():
    return (
        "You are the Outlier Detection Agent. "
        "Your task is to detect anomalous routes using the baseline dataset. "

        f"Input: '{OUTPUT_DIR}/baseline_data.csv'. "
        f"Output: '{OUTPUT_DIR}/outliers.csv'. "

        "Generate Python code using pandas, numpy, sklearn if available. "

        "STEP 1 — LOAD DATA: "
        "Load baseline_data.csv. "

        "STEP 2 — FEATURE ENGINEERING: "
        "Compute z_score using total_alarms: "
        "mean = df['total_alarms'].mean() "
        "std = df['total_alarms'].std() "
        "if std == 0 or NaN, set std = 1 "
        "df['z_score'] = (df['total_alarms'] - mean) / std "

        "Compute ratio_to_baseline: "
        "df['ratio_to_baseline'] = df['total_alarms'] / df['rolling_mean_alarms'] "

        "Replace inf with NaN. "

        "STEP 3 — ANOMALY SCORE: "
        "df['anomaly_score'] = "
        "  3 * df['ratio_to_baseline'].fillna(0) + "
        "  2 * df['total_alarms'].fillna(0) + "
        "  df['z_score'].abs().fillna(0) "

        "STEP 4 — TOP-K SELECTION: "
        "k = max(1, int(0.05 * len(df))) "
        "ranked = df.sort_values('anomaly_score', ascending=False) "
        "top_idx = ranked.head(k).index "

        "df['is_outlier'] = False "
        "df.loc[top_idx, 'is_outlier'] = True "

        "STEP 5 — OUTPUT: "
        "outliers = df[df['is_outlier'] == True].copy() "
        "Write outliers.csv. "

        "IMPORTANT: "
        "Always return at least one outlier. "
    )

In [512]:
# ── RUN: Outlier Detection Agent ─────────────────────────────
outlier_output_csv = os.path.join(OUTPUT_DIR, "outliers.csv")

result_outlier = run_agent_with_supervisor(
    task_name="outlier_detection",
    prompt=_build_outlier_prompt(),
    validator_fn=validate_outlier_output,
    output_path=outlier_output_csv,
    max_retries=3,
)

if result_outlier["succeeded"]:
    df_outliers = pd.read_csv(outlier_output_csv)
    print(f"\noutliers.csv: shape={df_outliers.shape}")
    print(df_outliers.head(10))


  AGENT: outlier_detection

  [attempt 1/4] temperature=0.0
  ── stdout ──
  │ Loading baseline_data.csv...
  │ Data loaded.
  │ Computing z_score...
  │ Computing ratio_to_baseline...
  │ Computing anomaly_score...
  │ Selecting top outliers...
  │ Writing outliers.csv...
  │ Outliers saved to outliers.csv.
  │ SUCCESS: At least one outlier detected.

  ┌─ Validation report attempt 1 ─
  │ ✓  file_exists            /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/outliers.csv
  │ ✓  loadable               shape=(23, 10)
  │ ✓  required_columns       all present: ['route', 'anomaly_score', 'is_outlier']
  │ ✓  non_empty              rows=23
  │ ✓  outliers_flagged       outliers=23
  └─ Overall: PASSED

  ✓ outlier_detection SUCCEEDED on attempt 1

outliers.csv: shape=(23, 10)
     route  rolling_mean_alarms  rolling_std_alarms  total_alarms  n_records  \
0  saw→bgy             1.000000            1.359449          80.0         80   
1  sa